In [ ]:
# FIGURE 2: UTrack vs ERA5 Total Column Water (TCW) Validation

import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.ticker import ScalarFormatter, FuncFormatter
import cartopy.crs as ccrs
import xskillscore as xs
import cmocean
import cmcrameri.cm as cmc

# =============================================================================
# 0. JOURNAL FORMATTING SETTINGS
# =============================================================================
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 12 
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['xtick.labelsize'] = 11
plt.rcParams['ytick.labelsize'] = 11
plt.rcParams['legend.fontsize'] = 11
plt.rcParams['axes.linewidth'] = 1.0 
plt.rcParams['pdf.fonttype'] = 42 
plt.rcParams['ps.fonttype'] = 42

def format_single_decimal(x, pos):
    return f"{x:.1f}"

# =============================================================================
# 1. CONFIGURATION
# =============================================================================
NOISE_THRESHOLD = 0.0

# ---> CONFIGURATION FOR TCW AVERAGE <---
DAYS_IN_MONTH = 31 
HOURS_PER_DAY = 24  

# Use relative paths assuming this script is run from the repository root
root_path = "./output/"

file_bwd = os.path.join(root_path, "UTrack_2024_05_bwd_Global_config_bwdmix24_interp.nc")
file_fwd = os.path.join(root_path, "UTrack_2024_05_fwd_Global_config_fwdmix24_interp.nc")

var_lag = 'tot_lagrangian_tcw'
var_era = 'tot_era5_tcw'
 
LABEL_UTRACK = "UTrack"
LABEL_ERA5 = "ERA5"

# =============================================================================
# 2. HELPER FUNCTIONS
# =============================================================================
def compute_grid_areas(lats, lons):
    R = 6371000.0 
    lats_np, lons_np = np.array(lats), np.array(lons)
    lat_bounds = np.zeros(len(lats_np) + 1)
    lat_bounds[1:-1] = 0.5 * (lats_np[:-1] + lats_np[1:])
    lat_bounds[0] = lats_np[0] - 0.5 * (lats_np[1] - lats_np[0])
    lat_bounds[-1] = lats_np[-1] + 0.5 * (lats_np[-1] - lats_np[-2])
    lat_bounds_rad = np.radians(lat_bounds)
    
    lon_bounds = np.zeros(len(lons_np) + 1)
    lon_bounds[1:-1] = 0.5 * (lons_np[:-1] + lons_np[1:])
    lon_bounds[0] = lons_np[0] - 0.5 * (lons_np[1] - lons_np[0])
    lon_bounds[-1] = lons_np[-1] + 0.5 * (lons_np[-1] - lons_np[-2])
    lon_bounds_rad = np.radians(lon_bounds)
    
    d_lon = np.abs(lon_bounds_rad[1:] - lon_bounds_rad[:-1]) 
    sin_lat = np.sin(lat_bounds_rad)
    d_sin_lat = np.abs(sin_lat[1:] - sin_lat[:-1])
    area_grid_np = R**2 * d_sin_lat[:, np.newaxis] * d_lon[np.newaxis, :]
    
    return xr.DataArray(area_grid_np, coords={'lat': lats, 'lon': lons}, dims=['lat', 'lon'])

def tcw_to_hourly_mean(filepath, v_lag, v_era, days, hours_per_day):
    """Extract data by transforming cumulative values into instantaneous hourly means."""
    with xr.open_dataset(filepath) as ds:
        if 'latitude' in ds.coords: ds = ds.rename({'latitude': 'lat', 'longitude': 'lon'})
        area_m2 = compute_grid_areas(ds.lat, ds.lon)
        area_safe = area_m2.where(area_m2 > 1e-6)
        total_hours = days * hours_per_day
        
        mm_l = (ds[v_lag] / area_safe).fillna(0) / total_hours
        mm_e = (ds[v_era] / area_safe).fillna(0) / total_hours
        vol_l = (ds[v_lag] / 1000.0).fillna(0) / total_hours
        vol_e = (ds[v_era] / 1000.0).fillna(0) / total_hours
        
        return mm_l, vol_l, mm_e, vol_e, area_m2

def compute_statistics(sim_xr, obs_xr, area_xr):
    weights = area_xr.fillna(0)
    mask = (obs_xr > 0)
    s_masked = sim_xr.where(mask)
    o_masked = obs_xr.where(mask)
    w_masked = weights.where(mask).fillna(0)
    
    sum_w = w_masked.sum().values.item()
    if sum_w == 0: return {'r': np.nan, 'RMSE': np.nan, 'MAE': np.nan, 'KGE': np.nan}
    
    r = xs.pearson_r(s_masked, o_masked, dim=['lat', 'lon'], weights=w_masked, skipna=True).values.item()
    rmse = xs.rmse(s_masked, o_masked, dim=['lat', 'lon'], weights=w_masked, skipna=True).values.item()
    mae = xs.mae(s_masked, o_masked, dim=['lat', 'lon'], weights=w_masked, skipna=True).values.item()
    
    mean_s = s_masked.weighted(w_masked).mean(dim=['lat', 'lon'], skipna=True).values.item()
    mean_o = o_masked.weighted(w_masked).mean(dim=['lat', 'lon'], skipna=True).values.item()
    std_s = s_masked.weighted(w_masked).std(dim=['lat', 'lon'], skipna=True).values.item()
    std_o = o_masked.weighted(w_masked).std(dim=['lat', 'lon'], skipna=True).values.item()
    
    if std_o == 0 or mean_o == 0 or np.isnan(r): kge = np.nan
    else: kge = 1 - np.sqrt((r - 1)**2 + (std_s / std_o - 1)**2 + (mean_s / mean_o - 1)**2)
    return {'r': r, 'RMSE': rmse, 'MAE': mae, 'KGE': kge}

def get_asymmetric_limits(data_array1, data_array2=None, margin=0.05):
    if data_array2 is not None:
        val_min = min(float(data_array1.min()), float(data_array2.min()))
        val_max = max(float(data_array1.max()), float(data_array2.max()))
    else:
        val_min, val_max = float(data_array1.min()), float(data_array1.max())
    span = val_max - val_min if val_max != val_min else 1.0
    return (val_min - span * margin, val_max + span * margin)

# =============================================================================
# 3. DATA PREPARATION
# =============================================================================
mm_l_bwd, vol_l_bwd, mm_e_bwd, vol_e_bwd, area_m2_bwd = tcw_to_hourly_mean(file_bwd, var_lag, var_era, DAYS_IN_MONTH, HOURS_PER_DAY)
mm_l_fwd, vol_l_fwd, mm_e_fwd, vol_e_fwd, area_m2_fwd = tcw_to_hourly_mean(file_fwd, var_lag, var_era, DAYS_IN_MONTH, HOURS_PER_DAY)

diff_mm_bwd = mm_l_bwd - mm_e_bwd
diff_mm_fwd = mm_l_fwd - mm_e_fwd

q98_bwd = float(xr.concat([mm_l_bwd.where(mm_l_bwd > 0), mm_e_bwd.where(mm_e_bwd > 0)], dim='z').quantile(0.98))
q98_fwd = float(xr.concat([mm_l_fwd.where(mm_l_fwd > 0), mm_e_fwd.where(mm_e_fwd > 0)], dim='z').quantile(0.98))
vmax_abs_global = max(q98_bwd, q98_fwd)

limit_diff_global = max(float(np.abs(diff_mm_bwd).quantile(0.98)), float(np.abs(diff_mm_fwd).quantile(0.98)))
if limit_diff_global == 0: limit_diff_global = 1.0

# Absolute values integration
lat_vol_l_raw_bwd, lat_vol_e_raw_bwd = vol_l_bwd.sum(dim='lon'), vol_e_bwd.sum(dim='lon')
lon_vol_l_bwd, lon_vol_e_bwd = vol_l_bwd.sum(dim='lat'), vol_e_bwd.sum(dim='lat')
lat_vol_l_bwd = lat_vol_l_raw_bwd.where(np.abs(lat_vol_l_raw_bwd.lat) <= 89, drop=True)
lat_vol_e_bwd = lat_vol_e_raw_bwd.where(np.abs(lat_vol_e_raw_bwd.lat) <= 89, drop=True)

lat_vol_l_raw_fwd, lat_vol_e_raw_fwd = vol_l_fwd.sum(dim='lon'), vol_e_fwd.sum(dim='lon')
lon_vol_l_fwd, lon_vol_e_fwd = vol_l_fwd.sum(dim='lat'), vol_e_fwd.sum(dim='lat')
lat_vol_l_fwd = lat_vol_l_raw_fwd.where(np.abs(lat_vol_l_raw_fwd.lat) <= 89, drop=True)
lat_vol_e_fwd = lat_vol_e_raw_fwd.where(np.abs(lat_vol_e_raw_fwd.lat) <= 89, drop=True)

vmax_zonal_abs = max(float(lat_vol_l_bwd.max()), float(lat_vol_e_bwd.max()), float(lat_vol_l_fwd.max()), float(lat_vol_e_fwd.max()))
xlim_zonal_abs = (0, vmax_zonal_abs * 1.05)

vmax_merid_abs = max(float(lon_vol_l_bwd.max()), float(lon_vol_e_bwd.max()), float(lon_vol_l_fwd.max()), float(lon_vol_e_fwd.max()))
ylim_merid_abs = (0, vmax_merid_abs * 1.05)

# Differences
diff_lat_vol_bwd = lat_vol_l_bwd - lat_vol_e_bwd
diff_lon_vol_bwd = lon_vol_l_bwd - lon_vol_e_bwd
diff_lat_vol_fwd = lat_vol_l_fwd - lat_vol_e_fwd
diff_lon_vol_fwd = lon_vol_l_fwd - lon_vol_e_fwd

xlim_zonal_diff = get_asymmetric_limits(diff_lat_vol_bwd, diff_lat_vol_fwd)
ylim_merid_diff = get_asymmetric_limits(diff_lon_vol_bwd, diff_lon_vol_fwd)

stats_bwd = compute_statistics(mm_l_bwd, mm_e_bwd, area_m2_bwd)
stats_fwd = compute_statistics(mm_l_fwd, mm_e_fwd, area_m2_fwd)

scale_factor_budget = 1e12
# BWD Budgets
tot_lag_km3_bwd = float(vol_l_bwd.sum().values) / scale_factor_budget
tot_era_km3_bwd = float(vol_e_bwd.sum().values) / scale_factor_budget
diff_km3_bwd = tot_lag_km3_bwd - tot_era_km3_bwd
perc_diff_bwd = (diff_km3_bwd / tot_era_km3_bwd) * 100 if tot_era_km3_bwd > 0 else np.nan
# FWD Budgets
tot_lag_km3_fwd = float(vol_l_fwd.sum().values) / scale_factor_budget
tot_era_km3_fwd = float(vol_e_fwd.sum().values) / scale_factor_budget
diff_km3_fwd = tot_lag_km3_fwd - tot_era_km3_fwd
perc_diff_fwd = (diff_km3_fwd / tot_era_km3_fwd) * 100 if tot_era_km3_fwd > 0 else np.nan

# =============================================================================
# 4. COLORMAPS & SCALERS
# =============================================================================
cmap_abs = cmc.glasgow_r.copy() 
cmap_abs.set_under('white')

COLOR_UTRACK_BWD = cmap_abs(0.85) 
COLOR_UTRACK_FWD = cmap_abs(0.85) 
COLOR_ERA5_BWD   = cmap_abs(0.25)
COLOR_ERA5_FWD   = cmap_abs(0.25)

cmap_diff = cmocean.cm.diff_r.copy() 
COLOR_POS = cmap_diff(0.85) 
COLOR_NEG = cmap_diff(0.15) 

scale_zonal_abs = 1e10 
label_zonal_abs = "10^1"
scale_merid_abs = 1e10 
label_merid_abs = "10^1"

scale_zonal_diff = 1e9
scale_merid_diff = 1e9

sf = ScalarFormatter(useOffset=False)
sf.set_scientific(False)
formatter_1d = FuncFormatter(format_single_decimal)

# =============================================================================
# 5. LAYOUT
# =============================================================================
fig_w, fig_h = 20.0, 12.0 
fig = plt.figure(figsize=(fig_w, fig_h)) 
proj = ccrs.PlateCarree()

w_map = 0.32            
h_map = (w_map * fig_w) / (2.0 * fig_h) 
w_zon = 0.08            
h_mer = 0.1             
h_cbar = 0.015          

# Left column (BWD)
x_left_c1 = 0.05
x_right_c1 = x_left_c1 + w_map + 0.01 

# Right column (FWD)
gap_cols = 0.08
x_left_c2 = x_right_c1 + w_zon + gap_cols
x_right_c2 = x_left_c2 + w_map + 0.01 

# Vertical coordinates
gap_map_mer = 0.01    
gap_mer_cbar = 0.05   
gap_blocks = 0.06    

# Top Row
y1_map_t = 0.95
y1_map_b = y1_map_t - h_map
y1_mer_t = y1_map_b - gap_map_mer
y1_mer_b = y1_mer_t - h_mer
y1_cbar_t = y1_mer_b - gap_mer_cbar
y1_cbar_b = y1_cbar_t - h_cbar

# Bottom Row
y2_map_t = y1_cbar_b - gap_blocks
y2_map_b = y2_map_t - h_map
y2_mer_t = y2_map_b - gap_map_mer
y2_mer_b = y2_mer_t - h_mer
y2_cbar_t = y2_mer_b - gap_mer_cbar
y2_cbar_b = y2_cbar_t - h_cbar

y1_box_top = y1_map_b - 0.055
h_box1 = y1_box_top - y1_mer_b

y2_box_top = y2_map_b - 0.055
h_box2 = y2_box_top - y2_mer_b

# ===========================================================================
# LEFT COLUMN (BACKWARD TCW)
# ===========================================================================
ax_m1_bwd = fig.add_axes([x_left_c1, y1_map_b, w_map, h_map], projection=proj)
ax_m1_bwd.set_rasterization_zorder(0)
im1_bwd = ax_m1_bwd.pcolormesh(mm_l_bwd.lon, mm_l_bwd.lat, mm_l_bwd.where(mm_l_bwd > 0.1), transform=proj, cmap=cmap_abs, vmin=0.0, vmax=vmax_abs_global, zorder=-1)
ax_m1_bwd.coastlines(linewidth=0.5, zorder=1)
ax_m1_bwd.set_extent([-180, 180, -90, 90], crs=proj)
ax_m1_bwd.text(-0.07, 1.0, 'a', transform=ax_m1_bwd.transAxes, fontsize=18, weight='bold', va='top')
for spine in ax_m1_bwd.spines.values(): spine.set_edgecolor('black'); spine.set_linewidth(1.0)

ax_z1_bwd = fig.add_axes([x_right_c1, y1_map_b, w_zon, h_map])
ax_z1_bwd.plot(lat_vol_l_bwd / scale_zonal_abs, lat_vol_l_bwd.lat, color=COLOR_UTRACK_BWD, linewidth=2.0, label=LABEL_UTRACK)
ax_z1_bwd.plot(lat_vol_e_bwd / scale_zonal_abs, lat_vol_e_bwd.lat, color=COLOR_ERA5_BWD, linestyle='--', linewidth=2.0, label=LABEL_ERA5)
ax_z1_bwd.set_xlim([x / scale_zonal_abs for x in xlim_zonal_abs]) 
ax_z1_bwd.set_ylim(-90, 90)
ax_z1_bwd.set_yticks(np.arange(-90, 91, 30))
ax_z1_bwd.yaxis.tick_right()
ax_z1_bwd.yaxis.set_label_position("right")
ax_z1_bwd.set_yticklabels([f"{abs(l)}°{'S' if l<0 else 'N' if l>0 else ''}" for l in np.arange(-90, 91, 30)])
ax_z1_bwd.set_ylabel("Latitude")
ax_z1_bwd.set_xlabel(f"Volume (${label_zonal_abs}$ m$^3$/h)")
ax_z1_bwd.xaxis.set_major_formatter(formatter_1d)
ax_z1_bwd.grid(True, linestyle=':', alpha=0.6)
ax_z1_bwd.spines['top'].set_visible(False)
ax_z1_bwd.spines['left'].set_visible(False)
ax_z1_bwd.legend(frameon=False, loc='upper right')

ax_me1_bwd = fig.add_axes([x_left_c1, y1_mer_b, w_map, h_mer])
ax_me1_bwd.plot(lon_vol_l_bwd.lon, lon_vol_l_bwd / scale_merid_abs, color=COLOR_UTRACK_BWD, linewidth=2.0)
ax_me1_bwd.plot(lon_vol_e_bwd.lon, lon_vol_e_bwd / scale_merid_abs, color=COLOR_ERA5_BWD, linestyle='--', linewidth=2.0)
ax_me1_bwd.set_ylim([y / scale_merid_abs for y in ylim_merid_abs]) 
ax_me1_bwd.set_xlim(-180, 180)
ax_me1_bwd.set_xticks(np.arange(-180, 181, 60))
ax_me1_bwd.set_xticklabels([f"{abs(l)}°{'W' if l<0 else 'E' if l>0 else ''}" for l in np.arange(-180, 181, 60)])
ax_me1_bwd.set_ylabel(f"Volume (${label_merid_abs}$ km$^3$/h)")
ax_me1_bwd.set_xlabel("Longitude")
ax_me1_bwd.yaxis.set_major_formatter(formatter_1d)
ax_me1_bwd.grid(True, linestyle=':', alpha=0.6)
ax_me1_bwd.spines['top'].set_visible(False)
ax_me1_bwd.spines['right'].set_visible(False)

ax_stat1_bwd = fig.add_axes([x_right_c1+0.02, y1_mer_b-0.02, w_zon-0.005, h_box1]) 
ax_stat1_bwd.axis('off')
budgets_text_bwd = (f"[$10^3$ km$^3$/h]\nUTrack: {tot_lag_km3_bwd:,.2f}\nERA5: {tot_era_km3_bwd:,.2f}")
ax_stat1_bwd.text(0.05, 0.95, budgets_text_bwd, transform=ax_stat1_bwd.transAxes, ha='left', va='top', fontsize=12)

cbar_ax1_bwd = fig.add_axes([x_left_c1, y1_cbar_b, w_map, h_cbar])
cb1_bwd = fig.colorbar(im1_bwd, cax=cbar_ax1_bwd, orientation='horizontal', extend='max')
cb1_bwd.solids.set_rasterized(True); cb1_bwd.solids.set_edgecolor('face'); cb1_bwd.outline.set_edgecolor('black')
cb1_bwd.set_label('Mean TCW (mm/h)', weight='bold')

# BWD Differences (Bottom)
ax_m2_bwd = fig.add_axes([x_left_c1, y2_map_b, w_map, h_map], projection=proj)
norm_diff = mcolors.TwoSlopeNorm(vmin=-limit_diff_global, vcenter=0.0, vmax=limit_diff_global)
ax_m2_bwd.set_rasterization_zorder(0)
im2_bwd = ax_m2_bwd.pcolormesh(diff_mm_bwd.lon, diff_mm_bwd.lat, diff_mm_bwd, transform=proj, cmap=cmap_diff, norm=norm_diff, zorder=-1)
ax_m2_bwd.coastlines(linewidth=0.5, zorder=1)
ax_m2_bwd.set_extent([-180, 180, -90, 90], crs=proj)
ax_m2_bwd.text(-0.07, 1.0, 'c', transform=ax_m2_bwd.transAxes, fontsize=18, weight='bold', va='top')
for spine in ax_m2_bwd.spines.values(): spine.set_edgecolor('black'); spine.set_linewidth(1.0)

ax_z2_bwd = fig.add_axes([x_right_c1, y2_map_b, w_zon, h_map])
ax_z2_bwd.axvline(x=0, color='gray', linestyle='--', linewidth=1.0, zorder=1)
diff_lat_scaled_bwd = diff_lat_vol_bwd / scale_zonal_diff
ax_z2_bwd.fill_betweenx(diff_lat_scaled_bwd.lat, 0, diff_lat_scaled_bwd, where=(diff_lat_scaled_bwd>0), color=COLOR_POS, alpha=0.6, zorder=2)
ax_z2_bwd.fill_betweenx(diff_lat_scaled_bwd.lat, 0, diff_lat_scaled_bwd, where=(diff_lat_scaled_bwd<0), color=COLOR_NEG, alpha=0.6, zorder=2)
ax_z2_bwd.plot(diff_lat_scaled_bwd, diff_lat_scaled_bwd.lat, color='black', linewidth=1.5, zorder=3)
ax_z2_bwd.set_xlim([x / scale_zonal_diff for x in xlim_zonal_diff]) 
ax_z2_bwd.set_ylim(-90, 90)
ax_z2_bwd.set_yticks(np.arange(-90, 91, 30))
ax_z2_bwd.yaxis.tick_right()
ax_z2_bwd.yaxis.set_label_position("right")
ax_z2_bwd.set_yticklabels([f"{abs(l)}°{'S' if l<0 else 'N' if l>0 else ''}" for l in np.arange(-90, 91, 30)])
ax_z2_bwd.set_ylabel("Latitude")
ax_z2_bwd.set_xlabel(f"$\\Delta$ Volume (km$^3$/h)")
ax_z2_bwd.xaxis.set_major_formatter(formatter_1d)
ax_z2_bwd.grid(True, linestyle=':', alpha=0.6)
ax_z2_bwd.spines['top'].set_visible(False)
ax_z2_bwd.spines['left'].set_visible(False)

ax_me2_bwd = fig.add_axes([x_left_c1, y2_mer_b, w_map, h_mer])
ax_me2_bwd.axhline(y=0, color='gray', linestyle='--', linewidth=1.0, zorder=1)
diff_lon_scaled_bwd = diff_lon_vol_bwd / scale_merid_diff
ax_me2_bwd.fill_between(diff_lon_scaled_bwd.lon, 0, diff_lon_scaled_bwd, where=(diff_lon_scaled_bwd>0), color=COLOR_POS, alpha=0.6, zorder=2)
ax_me2_bwd.fill_between(diff_lon_scaled_bwd.lon, 0, diff_lon_scaled_bwd, where=(diff_lon_scaled_bwd<0), color=COLOR_NEG, alpha=0.6, zorder=2)
ax_me2_bwd.plot(diff_lon_scaled_bwd.lon, diff_lon_scaled_bwd, color='black', linewidth=1.5, zorder=3)
ax_me2_bwd.set_ylim([y / scale_merid_diff for y in ylim_merid_diff])
ax_me2_bwd.set_xlim(-180, 180)
ax_me2_bwd.set_xticks(np.arange(-180, 181, 60))
ax_me2_bwd.set_xticklabels([f"{abs(l)}°{'W' if l<0 else 'E' if l>0 else ''}" for l in np.arange(-180, 181, 60)])
ax_me2_bwd.set_ylabel(f"$\\Delta$ Volume (km$^3$/h)")
ax_me2_bwd.set_xlabel("Longitude")
ax_me2_bwd.yaxis.set_major_formatter(formatter_1d)
ax_me2_bwd.grid(True, linestyle=':', alpha=0.6)
ax_me2_bwd.spines['top'].set_visible(False)
ax_me2_bwd.spines['right'].set_visible(False)

ax_stat2_bwd = fig.add_axes([x_right_c1+0.02, y2_mer_b-0.02, w_zon-0.005, h_box2]) 
ax_stat2_bwd.axis('off')
stats_text_bwd = (f"Pearson $r$: {stats_bwd['r']:.2f}\nRMSE: {stats_bwd['RMSE']:.2f} mm/h\nMAE: {stats_bwd['MAE']:.2f} mm/h\nKGE: {stats_bwd['KGE']:.2f}")
ax_stat2_bwd.text(0.05, 0.95, stats_text_bwd, transform=ax_stat2_bwd.transAxes, ha='left', va='top', fontsize=12)

cbar_ax2_bwd = fig.add_axes([x_left_c1, y2_cbar_b, w_map, h_cbar])
cb2_bwd = fig.colorbar(im2_bwd, cax=cbar_ax2_bwd, orientation='horizontal', extend='both')
cb2_bwd.solids.set_rasterized(True); cb2_bwd.solids.set_edgecolor('face'); cb2_bwd.outline.set_edgecolor('black')
cb2_bwd.set_label(f'Absolute Difference ({LABEL_UTRACK} - {LABEL_ERA5}) [mm/h]', weight='bold')

# ===========================================================================
# 2. RIGHT COLUMN (FORWARD TCW)
# ===========================================================================
ax_m1_fwd = fig.add_axes([x_left_c2, y1_map_b, w_map, h_map], projection=proj)
ax_m1_fwd.set_rasterization_zorder(0)
im1_fwd = ax_m1_fwd.pcolormesh(mm_l_fwd.lon, mm_l_fwd.lat, mm_l_fwd.where(mm_l_fwd > 0.1), transform=proj, cmap=cmap_abs, vmin=0.0, vmax=vmax_abs_global, zorder=-1)
ax_m1_fwd.coastlines(linewidth=0.5, zorder=1)
ax_m1_fwd.set_extent([-180, 180, -90, 90], crs=proj)
ax_m1_fwd.text(-0.07, 1.0, 'b', transform=ax_m1_fwd.transAxes, fontsize=18, weight='bold', va='top')
for spine in ax_m1_fwd.spines.values(): spine.set_edgecolor('black'); spine.set_linewidth(1.0)

ax_z1_fwd = fig.add_axes([x_right_c2, y1_map_b, w_zon, h_map])
ax_z1_fwd.plot(lat_vol_l_fwd / scale_zonal_abs, lat_vol_l_fwd.lat, color=COLOR_UTRACK_FWD, linewidth=2.0, label=LABEL_UTRACK)
ax_z1_fwd.plot(lat_vol_e_fwd / scale_zonal_abs, lat_vol_e_fwd.lat, color=COLOR_ERA5_FWD, linestyle='--', linewidth=2.0, label=LABEL_ERA5)
ax_z1_fwd.set_xlim([x / scale_zonal_abs for x in xlim_zonal_abs]) 
ax_z1_fwd.set_ylim(-90, 90)
ax_z1_fwd.set_yticks(np.arange(-90, 91, 30))
ax_z1_fwd.yaxis.tick_right()
ax_z1_fwd.yaxis.set_label_position("right")
ax_z1_fwd.set_yticklabels([f"{abs(l)}°{'S' if l<0 else 'N' if l>0 else ''}" for l in np.arange(-90, 91, 30)])
ax_z1_fwd.set_ylabel("Latitude")
ax_z1_fwd.set_xlabel(f"Volume (${label_zonal_abs}$ km$^3$/h)")
ax_z1_fwd.xaxis.set_major_formatter(formatter_1d)
ax_z1_fwd.grid(True, linestyle=':', alpha=0.6)
ax_z1_fwd.spines['top'].set_visible(False)
ax_z1_fwd.spines['left'].set_visible(False)
ax_z1_fwd.legend(frameon=False, loc='upper right')

ax_me1_fwd = fig.add_axes([x_left_c2, y1_mer_b, w_map, h_mer])
ax_me1_fwd.plot(lon_vol_l_fwd.lon, lon_vol_l_fwd / scale_merid_abs, color=COLOR_UTRACK_FWD, linewidth=2.0)
ax_me1_fwd.plot(lon_vol_e_fwd.lon, lon_vol_e_fwd / scale_merid_abs, color=COLOR_ERA5_FWD, linestyle='--', linewidth=2.0)
ax_me1_fwd.set_ylim([y / scale_merid_abs for y in ylim_merid_abs]) 
ax_me1_fwd.set_xlim(-180, 180)
ax_me1_fwd.set_xticks(np.arange(-180, 181, 60))
ax_me1_fwd.set_xticklabels([f"{abs(l)}°{'W' if l<0 else 'E' if l>0 else ''}" for l in np.arange(-180, 181, 60)])
ax_me1_fwd.set_ylabel(f"Volume (${label_merid_abs}$ km$^3$/h)")
ax_me1_fwd.set_xlabel("Longitude")
ax_me1_fwd.yaxis.set_major_formatter(formatter_1d)
ax_me1_fwd.grid(True, linestyle=':', alpha=0.6)
ax_me1_fwd.spines['top'].set_visible(False)
ax_me1_fwd.spines['right'].set_visible(False)

ax_stat1_fwd = fig.add_axes([x_right_c2+0.02, y1_mer_b-0.02, w_zon-0.01, h_box1]) 
ax_stat1_fwd.axis('off')
budgets_text_fwd = (f"[$10^3$ km$^3$/h]\nUTrack: {tot_lag_km3_fwd:,.2f}\nERA5: {tot_era_km3_fwd:,.2f}")
ax_stat1_fwd.text(0.05, 0.95, budgets_text_fwd, transform=ax_stat1_fwd.transAxes, ha='left', va='top', fontsize=12)

cbar_ax1_fwd = fig.add_axes([x_left_c2, y1_cbar_b, w_map, h_cbar])
cb1_fwd = fig.colorbar(im1_fwd, cax=cbar_ax1_fwd, orientation='horizontal', extend='max')
cb1_fwd.solids.set_rasterized(True); cb1_fwd.solids.set_edgecolor('face'); cb1_fwd.outline.set_edgecolor('black')
cb1_fwd.set_label('Mean TCW (mm/h)', weight='bold')

# FWD Differences (Bottom)
ax_m2_fwd = fig.add_axes([x_left_c2, y2_map_b, w_map, h_map], projection=proj)
ax_m2_fwd.set_rasterization_zorder(0)
im2_fwd = ax_m2_fwd.pcolormesh(diff_mm_fwd.lon, diff_mm_fwd.lat, diff_mm_fwd, transform=proj, cmap=cmap_diff, norm=norm_diff, zorder=-1)
ax_m2_fwd.coastlines(linewidth=0.5, zorder=1)
ax_m2_fwd.set_extent([-180, 180, -90, 90], crs=proj)
ax_m2_fwd.text(-0.07, 1.0, 'd', transform=ax_m2_fwd.transAxes, fontsize=18, weight='bold', va='top')
for spine in ax_m2_fwd.spines.values(): spine.set_edgecolor('black'); spine.set_linewidth(1.0)

ax_z2_fwd = fig.add_axes([x_right_c2, y2_map_b, w_zon, h_map])
ax_z2_fwd.axvline(x=0, color='gray', linestyle='--', linewidth=1.0, zorder=1)
diff_lat_scaled_fwd = diff_lat_vol_fwd / scale_zonal_diff
ax_z2_fwd.fill_betweenx(diff_lat_scaled_fwd.lat, 0, diff_lat_scaled_fwd, where=(diff_lat_scaled_fwd>0), color=COLOR_POS, alpha=0.6, zorder=2)
ax_z2_fwd.fill_betweenx(diff_lat_scaled_fwd.lat, 0, diff_lat_scaled_fwd, where=(diff_lat_scaled_fwd<0), color=COLOR_NEG, alpha=0.6, zorder=2)
ax_z2_fwd.plot(diff_lat_scaled_fwd, diff_lat_scaled_fwd.lat, color='black', linewidth=1.5, zorder=3)
ax_z2_fwd.set_xlim([x / scale_zonal_diff for x in xlim_zonal_diff]) 
ax_z2_fwd.set_ylim(-90, 90)
ax_z2_fwd.set_yticks(np.arange(-90, 91, 30))
ax_z2_fwd.yaxis.tick_right()
ax_z2_fwd.yaxis.set_label_position("right")
ax_z2_fwd.set_yticklabels([f"{abs(l)}°{'S' if l<0 else 'N' if l>0 else ''}" for l in np.arange(-90, 91, 30)])
ax_z2_fwd.set_ylabel("Latitude")
ax_z2_fwd.set_xlabel(f"$\\Delta$ Volume (km$^3$/h)")
ax_z2_fwd.xaxis.set_major_formatter(formatter_1d)
ax_z2_fwd.grid(True, linestyle=':', alpha=0.6)
ax_z2_fwd.spines['top'].set_visible(False)
ax_z2_fwd.spines['left'].set_visible(False)

ax_me2_fwd = fig.add_axes([x_left_c2, y2_mer_b, w_map, h_mer])
ax_me2_fwd.axhline(y=0, color='gray', linestyle='--', linewidth=1.0, zorder=1)
diff_lon_scaled_fwd = diff_lon_vol_fwd / scale_merid_diff
ax_me2_fwd.fill_between(diff_lon_scaled_fwd.lon, 0, diff_lon_scaled_fwd, where=(diff_lon_scaled_fwd>0), color=COLOR_POS, alpha=0.6, zorder=2)
ax_me2_fwd.fill_between(diff_lon_scaled_fwd.lon, 0, diff_lon_scaled_fwd, where=(diff_lon_scaled_fwd<0), color=COLOR_NEG, alpha=0.6, zorder=2)
ax_me2_fwd.plot(diff_lon_scaled_fwd.lon, diff_lon_scaled_fwd, color='black', linewidth=1.5, zorder=3)
ax_me2_fwd.set_ylim([y / scale_merid_diff for y in ylim_merid_diff])
ax_me2_fwd.set_xlim(-180, 180)
ax_me2_fwd.set_xticks(np.arange(-180, 181, 60))
ax_me2_fwd.set_xticklabels([f"{abs(l)}°{'W' if l<0 else 'E' if l>0 else ''}" for l in np.arange(-180, 181, 60)])
ax_me2_fwd.set_ylabel(f"$\\Delta$ Volume (km$^3$/h)")
ax_me2_fwd.set_xlabel("Longitude")
ax_me2_fwd.yaxis.set_major_formatter(formatter_1d)
ax_me2_fwd.grid(True, linestyle=':', alpha=0.6)
ax_me2_fwd.spines['top'].set_visible(False)
ax_me2_fwd.spines['right'].set_visible(False)

ax_stat2_fwd = fig.add_axes([x_right_c2+0.02, y2_mer_b-0.02, w_zon-0.01, h_box2])
ax_stat2_fwd.axis('off') 
stats_text_fwd = (f"Pearson $r$: {stats_fwd['r']:.2f}\nRMSE: {stats_fwd['RMSE']:.2f} mm/h\nMAE: {stats_fwd['MAE']:.2f} mm/h\nKGE: {stats_fwd['KGE']:.2f}")
ax_stat2_fwd.text(0.05, 0.95, stats_text_fwd, transform=ax_stat2_fwd.transAxes, ha='left', va='top', fontsize=12)

cbar_ax2_fwd = fig.add_axes([x_left_c2, y2_cbar_b, w_map, h_cbar])
cb2_fwd = fig.colorbar(im2_fwd, cax=cbar_ax2_fwd, orientation='horizontal', extend='both')
cb2_fwd.solids.set_rasterized(True); cb2_fwd.solids.set_edgecolor('face'); cb2_fwd.outline.set_edgecolor('black')
cb2_fwd.set_label(f'Absolute Difference ({LABEL_UTRACK} - {LABEL_ERA5}) [mm/h]', weight='bold')

#plt.savefig('fig2.pdf', format='pdf', bbox_inches='tight', dpi=600)
plt.show()